<a id="top"></a>

# Lab L0404: build a prompt-chaining workflow

```yaml
title: "Lab L0404: build a prompt-chaining workflow"
keywords: langgraph, stategraph, prompt chaining, workflow, dag, typed state, reducer, node, edge, compile, invoke, lab
estimated duration: 35
```

> **Lesson:** L04. See [objectives.md](../../../../docs/origin/lesson_roadmaps/L04/objectives.md).
> **This is the solutions key** for `L0404_lab_empty.ipynb`.
> Runs **fully offline** -- a deterministic `StubChat` stands in for `ChatAnthropic` so you can focus
> on wiring the graph. No API key needed.

## Contents

- [1. Setup](#1-setup)
- [2. Problem 1 -- The typed state](#2-problem-1----the-typed-state)
- [3. Problem 2 -- The three nodes](#3-problem-2----the-three-nodes)
- [4. Problem 3 -- Wire, compile, render](#4-problem-3----wire-compile-render)
- [5. Problem 4 -- Run the workflow](#5-problem-4----run-the-workflow)
- [6. Problem 5 -- From stub to real model (written)](#6-problem-5----from-stub-to-real-model-written)
- [7. Self-check](#7-self-check)

## 1. Setup

Everything below is **given**: the `StubChat`/`StubReply` offline stand-in for `ChatAnthropic`, two
sample-ticket constants, the `POLICY` text, and two stub clients (`haiku`, `sonnet`). You build the
graph. Run this cell first.

In [1]:
from operator import add
from typing import Annotated, TypedDict

from langgraph.graph import END, StateGraph

HAIKU = "claude-haiku-4-5"  # the names are real; here a stub stands in for the client
SONNET = "claude-sonnet-4-6"

TICKETS = {
    "billing": "I was charged twice for my subscription this month -- please refund the extra charge.",
    "technical": "The export button throws a 500 error every time I click it on the reports page.",
    "general": "Do you offer a student discount, and how would I apply it to my account?",
}
POLICY = (
    "Refunds: a duplicate charge is always refundable within 60 days. "
    "Never promise a refund for change-of-mind. "
    "Escalate anything mentioning legal action or chargebacks to a human."
)


class StubReply:
    """Mimics the one field we read off a ChatAnthropic response: `.content`."""

    def __init__(self, content: str) -> None:
        self.content = content


class StubChat:
    """A tiny OFFLINE stand-in for ChatAnthropic, so this lab is free, fast, and deterministic.

    It mimics the single call our nodes use -- `.invoke(prompt).content` -> str. The graph
    wiring you practice here is identical with a real model: swapping `StubChat(HAIKU)` for
    `ChatAnthropic(model=HAIKU, api_key=require_anthropic_key())` is a one-line change, and the
    node code never changes (Problem 5). The lecture demos use the real client.
    """

    def __init__(self, model: str) -> None:
        self.model = model

    def invoke(self, prompt: str) -> StubReply:
        p = prompt.lower()
        if "classify" in p:
            # Key on TICKET-content words only. We avoid "bill"/"technical"/"general" because
            # those leak from the instruction text itself ("billing, technical, or general").
            if any(w in p for w in ("charge", "charged", "refund", "invoice", "payment", "twice")):
                return StubReply("billing")
            if any(w in p for w in ("error", "500", "crash", "bug", "broken", "login", "log in")):
                return StubReply("technical")
            return StubReply("general")
        if "summarize" in p:
            return StubReply("The customer needs help resolving an account issue.")
        if "compliance" in p or "policy" in p:
            return StubReply("OK: the reply is consistent with the refund policy.")
        return StubReply("Thanks for reaching out -- happy to help you with this!")


# Per-node models -- a stub each, exactly where a real graph would bind Haiku vs. Sonnet.
haiku = StubChat(HAIKU)
sonnet = StubChat(SONNET)

[↑ Back to top](#top)

## 2. Problem 1 -- The typed state

Define the **state** the chain threads through its nodes. It needs the raw `ticket`, the `parsed`
summary, the `draft`, the `verdict`, and a `steps` list. Make `steps` use the **append reducer**
(`Annotated[list[str], add]`) so each node *adds* its name; the other fields overwrite by default.

In [2]:
class TicketState(TypedDict):
    """State threaded through the parse -> draft -> policy_check chain."""

    ticket: str  # the raw incoming ticket (the input)
    parsed: str  # one-line issue summary from the parse node
    draft: str  # the reply from the draft node
    verdict: str  # the policy-check node's ruling
    # `steps` ACCUMULATES via the `add` reducer; the others overwrite (last write wins).
    steps: Annotated[list[str], add]

[↑ Back to top](#top)

## 3. Problem 2 -- The three nodes

Write the three nodes. Each reads the state and returns a **partial update** (only the fields it
changed), never the whole state. The model lives *inside* these nodes -- `parse` on the cheap stub
(`haiku`), `draft` and `policy_check` on the capable stub (`sonnet`). Each also appends its name to
`steps`.

In [3]:
def parse(state: TicketState) -> dict[str, object]:
    """Summarize the ticket's core issue. A LIGHT step -> Haiku."""
    reply = haiku.invoke(f"In one sentence, summarize this ticket's issue:\n\n{state['ticket']}")
    return {"parsed": str(reply.content), "steps": ["parse"]}


def draft(state: TicketState) -> dict[str, object]:
    """Write a customer reply from the parsed issue. The HEAVY step -> Sonnet."""
    reply = sonnet.invoke(
        f"Write a short, friendly support reply for this issue:\n\n{state['parsed']}"
    )
    return {"draft": str(reply.content), "steps": ["draft"]}


def policy_check(state: TicketState) -> dict[str, object]:
    """Check the draft against the policy snippet. A reasoning step -> Sonnet."""
    reply = sonnet.invoke(
        "You are a compliance reviewer. Reply 'OK: <reason>' or 'REVISE: <reason>'.\n"
        f"Policy:\n{POLICY}\n\nDraft:\n{state['draft']}"
    )
    return {"verdict": str(reply.content), "steps": ["policy_check"]}

[↑ Back to top](#top)

## 4. Problem 3 -- Wire, compile, render

Wire the fixed chain **parse -> draft -> policy_check -> END** with the `StateGraph` builder,
`compile()` it to `chain_app`, and print the diagram with `draw_mermaid()`. Notice: the render needs
no model at all -- the control flow is just **data**.

In [4]:
builder = StateGraph(TicketState)
builder.add_node("parse", parse)
builder.add_node("draft", draft)
builder.add_node("policy_check", policy_check)
builder.set_entry_point("parse")
builder.add_edge("parse", "draft")
builder.add_edge("draft", "policy_check")
builder.add_edge("policy_check", END)
chain_app = builder.compile()

# Control flow as data: a DAG, every edge moves forward to END -- no back-edge, so it's a workflow.
print(chain_app.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	parse(parse)
	draft(draft)
	policy_check(policy_check)
	__end__([<p>__end__</p>]):::last
	__start__ --> parse;
	draft --> policy_check;
	parse --> draft;
	policy_check --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



[↑ Back to top](#top)

## 5. Problem 4 -- Run the workflow

`invoke()` the graph on the billing ticket and print the `steps` and the `draft`. Run it twice: the
**path is identical every time** -- you wired it, so it's deterministic. (The stub's text is fixed;
with a real model the *wording* would vary but the *path* would not.)

In [5]:
result = chain_app.invoke({"ticket": TICKETS["billing"], "steps": []})
print("path  :", result["steps"])  # always ['parse', 'draft', 'policy_check']
print("draft :", result["draft"])
print("verdict:", result["verdict"])

path  : ['parse', 'draft', 'policy_check']
draft : Thanks for reaching out -- happy to help you with this!
verdict: OK: the reply is consistent with the refund policy.


[↑ Back to top](#top)

## 6. Problem 5 -- From stub to real model (written)

The lecture demo used the real `ChatAnthropic` client; this lab used `StubChat`. In a sentence or
two: **which line(s) would you change** to make these nodes call a real model, and **why does the
node code itself not change**? (Hint: what interface do `StubChat` and `ChatAnthropic` share?)

_Write your answer by editing this cell (double-click)._

_TODO_

[↑ Back to top](#top)

## 7. Self-check

Compare against `L0404_lab_solutions.ipynb`. You're done when: `TicketState` has the five fields
with an `add` reducer on `steps`; the three nodes each return a partial update and append to `steps`;
`chain_app` compiles and `draw_mermaid()` shows `parse -> draft -> policy_check -> END`; invoking
yields `steps == ['parse', 'draft', 'policy_check']` every run; and you can say why swapping the
client doesn't touch the node code (both expose `.invoke(prompt).content`).

[↑ Back to top](#top)